# United States Labor Force Data Dashboard

This dashboard was created as part of the SIADS 521 course at the University of Michigan. The purpose of the dashboard is for the user to explore United States labor force data while learning about how to build a dashboard in python. 

# Visualization Library

The plots for this dashboard will be created with Plotly Express. Plotly Express is an API for Plotly that allows the user to set many variables for each plot type, but is simpler than using the Plotly Graph Objects API. Plotly Express has built in hover notation showing useful information about the datapoint the cursor is above. It also allows the user to pan and zoom in any visualization to better explore the data than is possible with a fixed visualization. Graph Objects would be useful for a dashboard for external use where the ability to control every aspect of the visualization would be beneficial for corporate branding reasons, but does not seem necessary for an exploratory data analysis dashboard. I chose Plotly over Bokeh visualizations in part because the my team at work uses Plotly to make interactive visualizations when needed, so I have a little bit of familiarity with it. 

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly 
import ipywidgets as widgets
from fredapi import Fred
import datetime as dt
from IPython.display import display

### Import Data from Federal Reserve Bank of St. Louis

In [2]:
fred = Fred(api_key='4f2097fffd781e31f4f401937d28fd23')

In [3]:
series_dict = {"Job Openings (Total Nonfarm)": "JTSJOL",
               "Hires (Total Nonfarm)": "JTSHIL",
               "Quits (Total Nonfarm)": "JTSQUL",
               "Layoffs & Discharges (Total Nonfarm)": "JTSLDL",
               "Total Separations (Total Nonfarm)": "JTSTSL",
               "Unemployment Rate": "UNRATE",
               "Labor Force Participation Rate": "CIVPART",
               "Employment Level": "CE16OV",
               "Unemployment Level": "UNEMPLOY",
               "Job Openings in South Census Region": "JTS00SOJOL",
               "Job Openings in Midwest Census Region": "JTS00MWJOL",
               "Job Openings in West Census Region": "JTS00WEJOL", 
               "Job Openings in Northeast Census Region": "JTS00NEJOL",
               "Hires in South Census Region": "JTS00SOHIL",
               "Hires in Midwest Census Region": "JTS00MWHIL",
               "Hires in West Census Region": "JTS00WEHIL",
               "Hires in Northeast Census Region": "JTS00NEHIL",
               "Total Separations in South Census Region": "JTS00SOTSL",
               "Total Separations in Midwest Census Region": "JTS00MWTSL",
               "Total Separations in West Census Region": "JTS00WETSL",
               "Total Separations in Northeast Census Region": "JTS00NETSL"
              }

In [4]:
start_date = "2001-01-01"
first = 1
for key, value in series_dict.items():
    temp = fred.get_series(value, observation_start=start_date)
    temp = temp.rename(key).reset_index().rename(columns={"index": "DateTime"})
    temp["DateTime"] = pd.to_datetime(temp["DateTime"])
    if first == 1:
        jobs = temp
        first = 0
    jobs[key] = temp[key]

In [5]:
jobs['Date'] = jobs['DateTime'].dt.date
jobs['Year'] = jobs['DateTime'].dt.year
jobs['Month'] = jobs['DateTime'].dt.month
jobs['date_num'] = jobs['DateTime'].astype('int64')

In [6]:
levels = ["Job Openings (Total Nonfarm)", "Hires (Total Nonfarm)", "Quits (Total Nonfarm)",
         "Layoffs & Discharges (Total Nonfarm)", "Total Separations (Total Nonfarm)",
         "Employment Level", "Unemployment Level"]
scatter_choices = levels + ["Unemployment Rate", "Labor Force Participation Rate"]
openings = ["Job Openings in South Census Region", "Job Openings in Midwest Census Region",
            "Job Openings in West Census Region", "Job Openings in Northeast Census Region"]
hires = ["Hires in South Census Region", "Hires in Midwest Census Region",
         "Hires in West Census Region", "Hires in Northeast Census Region"]
separations = ["Total Separations in South Census Region", "Total Separations in Midwest Census Region",
               "Total Separations in West Census Region", "Total Separations in Northeast Census Region"]
bar_choices = ["Job Openings", "Hires", "Total Separations"]
init_lineplot =  ['Job Openings (Total Nonfarm)', 'Hires (Total Nonfarm)','Quits (Total Nonfarm)',
                  'Layoffs & Discharges (Total Nonfarm)', 'Total Separations (Total Nonfarm)']
bar_map = {"Job Openings": openings, "Hires": hires, "Total Separations": separations}

# Visualization Technique

This dashboard uses four different visualization techniques to help the user explore the data. Each of these visualizations are built using reusable functions as that simplifies implementation in the dashboard and will allow this notebook to be reused when developing additional dashboards. 

## Line Plot

The line plot will plot several variables (y-axis) over time (x-axis). This plot will be used to show "level" variables, which all have units of 1000 times the value. This type of plot is useful for showing trends in data over time.

In [7]:
def create_lineplot(data, var_names, title=None, size=[800,500]):
    """
    Creates a lineplot of multiple variables over time

    Parameters:
    data : pandas DataFrame
        The dataframe with the data to plot, must have a column named "Date"
    var_names : list of strings
        List with strings of column names to plot
    title : string
        Plot title (optional)
    size : list of integers
        List with image width and height in pixels
    """
    
    fig = px.line(data, "Date", var_names, 
                  title=title or 'Labor Force Levels Over Time', 
                  width=size[0],
                  height=size[1]
                 )
    fig.update_layout(yaxis_title='Value x 1000',
                     margin=dict(t=40, b=40, l=40, r=40),
                     legend=dict(title='Variable', orientation='h', yanchor='top', y=-0.15))
    return fig

In [8]:
lineplot = create_lineplot(jobs, init_lineplot)
lineplot.show()

## Scatter Plot

In [9]:
def create_scatterplot(data, x_var, y_var, title=None, size=[800,500]):
    """
    Creates a scatter of two variables to look for a relationships between them.
    The date will be used to color the points of the plot

    Parameters:
    data : pandas DataFrame
        The dataframe with the data to plot, must have a column named "Date"
    x_var : string
        String of column name to plot on x-axis
    y_var : string
        String of column name to plot on y-axis
    color : string
        String of column name to use to color the dots of the plot
    title : string
        Plot title (optional)
    size : list of integers
        List with image width and height in pixels
    """
    tick_vals = np.linspace(data['date_num'].min(), data['date_num'].max(), 6)
    tick_text = pd.to_datetime(tick_vals).strftime('%Y-%m')
    
    fig = px.scatter(data, x_var, y_var, color="date_num", 
                     color_continuous_scale='Viridis',
                     title=title or f'{y_var} vs. {x_var}', 
                     width=size[0],
                     height=size[1],
                     hover_data={'Date': True, 'date_num': False}
                     )
    fig.update_layout(margin=dict(t=40, b=40, l=40, r=40))
    fig.update_coloraxes(colorbar_title='Date',
                         colorbar_tickvals=tick_vals,
                         colorbar_ticktext=tick_text)
    return fig

In [10]:
scatter = create_scatterplot(jobs, 'Job Openings (Total Nonfarm)', 'Unemployment Level')
scatter.show()

## Stacked Bar Chart

In [11]:
def create_stackedbarchart(data, var_names, title=None, size=[800,500]):
    """
    Creates a stacked bar chart of one of 3 sets of region based labor data.
    Function assumes there is a "Year" column in the data as it sums all values for each year
    
    Parameters:
    data : pandas DataFrame
        The dataframe with the data to plot, must have a column named "Year"
    var_names : list of strings
        List with strings of column names to plot
    title : string
        Plot title (optional)
    size : list of integers
        List with image width and height in pixels
    """
    columns = ['Year'] + var_names
    annual = data[columns].groupby('Year', as_index=False).sum()
    
    fig = px.bar(annual, x='Year', y=var_names, 
                 width=size[0], height=size[1],
                 title=title or "Annual Regional Labor Data"
                )

    fig.update_layout(yaxis_title='Value x 1000',
                     margin=dict(t=40, b=40, l=40, r=40),
                     legend=dict(title='Value', orientation='h', yanchor='top', y=-0.15))
    return fig

In [12]:
barchart = create_stackedbarchart(jobs, hires)
barchart.show()

## Heat Map

In [13]:
def create_heatmap(data, variable, title=None, size=[500,800]):
    """
    Creates a heatmap of a given variable where each row is a year and 
    each column is a month. 
    
    Parameters:
    data : pandas DataFrame
        The dataframe with the data to plot, must have a columns named "Month" and "Year"
    variable : string
        String of column name to plot
    title : string
        Plot title (optional)
    size : list of integers
        List with image width and height in pixels
    """
    pivot = data.pivot_table(index='Year', columns='Month', values=variable)
    fig = px.imshow(pivot, color_continuous_scale='Viridis',
                    width=size[0], height=size[1],
                    title=title or f'{variable} Heat Map: Year x Month'
                   )
    return fig


In [14]:
heatmap = create_heatmap(jobs, 'Unemployment Rate')
heatmap.show()

### Widgets

In [15]:
year_range = widgets.IntRangeSlider(
    value=[2001,2026],
    min=2001, max=2026, step=1,
    continuous_update=False,
    style={'description_width': 'initial'}
)

level_picker = widgets.SelectMultiple(
    options=levels,
    value=init_lineplot,
    rows=6,
    style={'description_width': 'initial'}
)

scatter_x = widgets.Dropdown(
    options=scatter_choices,
    value='Job Openings (Total Nonfarm)',
    style={'description_width': 'initial'}
)

scatter_y = widgets.Dropdown(
    options=scatter_choices,
    value='Unemployment Level',
    style={'description_width': 'initial'}
)

bar_choice = widgets.Dropdown(
    options=bar_choices,
    value='Job Openings',
    style={'description_width': 'initial'}
)

heatmap_choice = widgets.Dropdown(
    options=scatter_choices,
    value='Unemployment Rate',
    style={'description_width': 'initial'}
)

out_line    = widgets.Output()
out_scatter = widgets.Output()
out_bar     = widgets.Output()
out_heatmap = widgets.Output()

In [16]:
def year_filter_data():
    y_min, y_max = year_range.value
    return jobs[(jobs['Year'] >= y_min) & (jobs['Year'] <= y_max)].copy()

In [17]:
def update_all(change=None):
    df = year_filter_data()

    with out_line:
        out_line.clear_output(wait=True)
        cols = list(level_picker.value)
        if cols:
            create_lineplot(df, cols).show()
        else:
            print("Select at least one series for the line plot")

    with out_scatter:
        out_scatter.clear_output(wait=True)
        create_scatterplot(df, scatter_x.value, scatter_y.value).show()

    with out_bar:
        out_bar.clear_output(wait=True)
        measure = bar_choice.value
        create_stackedbarchart(df, bar_map[measure],
                               title=f"Annual {measure} by Census Region").show()

    with out_heatmap:
        out_heatmap.clear_output(wait=True)
        create_heatmap(df, heatmap_choice.value).show()

In [18]:
for w in (year_range, level_picker, scatter_x, scatter_y, bar_choice, heatmap_choice):
    w.observe(update_all, names="value")

In [19]:
header = widgets.HTML(
    "<h1>U.S. Labor Market Dashboard</h1>"
    "<p>Source: BLS JOLTS via FRED (Federal Reserve Bank of St. Louis)</p>"
)


controls = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>Years</b>"), year_range]),
    widgets.VBox([widgets.HTML("<b>Line Plot Data</b>"), level_picker]),
    widgets.VBox([widgets.HTML("<b>Scatter Plot Data</b>"), scatter_x, scatter_y]),
    widgets.VBox([widgets.HTML("<b>Bar Chart Data</b>"), bar_choice]),
    widgets.VBox([widgets.HTML("<b>Heat Map Data</b>"), heatmap_choice]),
])


grid = widgets.GridspecLayout(2, 2, height="1050px", width="100%")
grid[0, 0] = out_line     
grid[0, 1] = out_scatter    
grid[1, 0] = out_bar        
grid[1, 1] = out_heatmap    


dashboard = widgets.VBox([
    header,
    controls,
    grid
])


update_all()
display(dashboard)